In [6]:
# W8D2 - RAG Evaluation with Ragas
# Cell 1: Install required packages

!pip install -q -U \
    langchain \
    langchain-community \
    langchain-chroma \
    chromadb \
    sentence-transformers \
    ragas \
    datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.7/163.7 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.

In [7]:
# Cell 2: Imports

import os
import pandas as pd

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

print("All imports completed successfully.")

All imports completed successfully.


/tmp/ipykernel_1385/1942197850.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings


In [8]:
# Cell 3: Create source documents

documents = [
    Document(page_content="""
Artificial Intelligence (AI) is a field of computer science that focuses on
creating systems that can perform tasks that normally require human
intelligence. These tasks include learning, reasoning, problem solving,
perception, and understanding language.
"""),

    Document(page_content="""
Machine Learning (ML) is a subset of Artificial Intelligence. It enables
computers to learn patterns from data and make predictions or decisions
without being explicitly programmed for every individual task.
"""),

    Document(page_content="""
Supervised learning is a machine learning approach in which a model learns
from labelled training data. Classification and regression are common
supervised learning tasks.
"""),

    Document(page_content="""
Unsupervised learning uses data without labelled outputs. Clustering is a
common unsupervised learning technique that groups similar data points
together.
"""),

    Document(page_content="""
Deep Learning is a branch of machine learning that uses neural networks with
multiple layers. It is commonly used for image recognition, speech
recognition, and natural language processing.
"""),

    Document(page_content="""
Natural Language Processing (NLP) is a field of Artificial Intelligence that
allows computers to process and understand human language. Applications
include translation, sentiment analysis, chatbots, and text summarization.
"""),

    Document(page_content="""
Large Language Models (LLMs) are machine learning models trained on large
amounts of text. They can perform tasks such as question answering,
summarization, text generation, and translation.
"""),

    Document(page_content="""
Retrieval-Augmented Generation (RAG) combines information retrieval with
language generation. A RAG system retrieves relevant information and gives
that information to a language model as context for generating an answer.
"""),

    Document(page_content="""
Vector databases store numerical representations called embeddings.
Embeddings allow systems to perform similarity searches and retrieve
documents that are semantically related to a query.
"""),

    Document(page_content="""
ChromaDB is a vector database commonly used in AI and RAG applications.
It stores document embeddings and supports similarity-based retrieval.
""")
]

print("Number of source documents:", len(documents))

Number of source documents: 10


In [9]:
# Cell 4: Create document chunks

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks[:5]):
    print(f"\n--- Chunk {i + 1} ---")
    print(chunk.page_content)

Number of chunks: 10

--- Chunk 1 ---
Artificial Intelligence (AI) is a field of computer science that focuses on
creating systems that can perform tasks that normally require human
intelligence. These tasks include learning, reasoning, problem solving,
perception, and understanding language.

--- Chunk 2 ---
Machine Learning (ML) is a subset of Artificial Intelligence. It enables
computers to learn patterns from data and make predictions or decisions
without being explicitly programmed for every individual task.

--- Chunk 3 ---
Supervised learning is a machine learning approach in which a model learns
from labelled training data. Classification and regression are common
supervised learning tasks.

--- Chunk 4 ---
Unsupervised learning uses data without labelled outputs. Clustering is a
common unsupervised learning technique that groups similar data points
together.

--- Chunk 5 ---
Deep Learning is a branch of machine learning that uses neural networks with
multiple layers. It is com

In [10]:
# Cell 5: Create ChromaDB vector store

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="w8d2_rag"
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

print("ChromaDB created successfully.")
print("Retriever k:", 3)
print("Embedding model: all-MiniLM-L6-v2")

/tmp/ipykernel_1385/216148162.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ChromaDB created successfully.
Retriever k: 3
Embedding model: all-MiniLM-L6-v2


In [11]:
# Cell 6: Create 10 evaluation questions with reference answers

qa_data = [
    {
        "question": "What is Artificial Intelligence?",
        "reference": "Artificial Intelligence is a field of computer science that creates systems capable of tasks that normally require human intelligence."
    },
    {
        "question": "What is Machine Learning?",
        "reference": "Machine Learning is a subset of Artificial Intelligence that enables computers to learn patterns from data and make predictions or decisions."
    },
    {
        "question": "What is supervised learning?",
        "reference": "Supervised learning is a machine learning approach where a model learns from labelled training data."
    },
    {
        "question": "What is unsupervised learning?",
        "reference": "Unsupervised learning uses data without labelled outputs and can group similar data points using techniques such as clustering."
    },
    {
        "question": "What is Deep Learning?",
        "reference": "Deep Learning is a branch of machine learning that uses neural networks with multiple layers."
    },
    {
        "question": "What is Natural Language Processing?",
        "reference": "Natural Language Processing is a field of Artificial Intelligence that enables computers to process and understand human language."
    },
    {
        "question": "What are Large Language Models?",
        "reference": "Large Language Models are machine learning models trained on large amounts of text that can perform tasks such as question answering and text generation."
    },
    {
        "question": "What is Retrieval-Augmented Generation?",
        "reference": "Retrieval-Augmented Generation combines information retrieval with language generation by providing retrieved information as context to a language model."
    },
    {
        "question": "What is a vector database?",
        "reference": "A vector database stores numerical representations called embeddings and supports similarity searches."
    },
    {
        "question": "What is ChromaDB?",
        "reference": "ChromaDB is a vector database commonly used in AI and RAG applications to store embeddings and perform similarity-based retrieval."
    }
]

qa_df = pd.DataFrame(qa_data)

print("Number of Q&A pairs:", len(qa_df))
qa_df

Number of Q&A pairs: 10


,question,reference
0,What is Artificial Intelligence?,Artificial Intelligence is a field of computer...
1,What is Machine Learning?,Machine Learning is a subset of Artificial Int...
2,What is supervised learning?,Supervised learning is a machine learning appr...
3,What is unsupervised learning?,Unsupervised learning uses data without labell...
4,What is Deep Learning?,Deep Learning is a branch of machine learning ...
5,What is Natural Language Processing?,Natural Language Processing is a field of Arti...
6,What are Large Language Models?,Large Language Models are machine learning mod...
7,What is Retrieval-Augmented Generation?,Retrieval-Augmented Generation combines inform...
8,What is a vector database?,A vector database stores numerical representat...
9,What is ChromaDB?,ChromaDB is a vector database commonly used in...


In [12]:
# Cell 7: Retrieve relevant contexts for all 10 questions

retrieved_contexts = []

for question in qa_df["question"]:
    docs = retriever.invoke(question)

    contexts = [doc.page_content.strip() for doc in docs]
    retrieved_contexts.append(contexts)

qa_df["retrieved_contexts"] = retrieved_contexts

print("Retrieved contexts for all questions.")

print("\nExample:")
print("Question:", qa_df.loc[0, "question"])
print("\nRetrieved context:")
print(qa_df.loc[0, "retrieved_contexts"][0])

Retrieved contexts for all questions.

Example:
Question: What is Artificial Intelligence?

Retrieved context:
Artificial Intelligence (AI) is a field of computer science that focuses on
creating systems that can perform tasks that normally require human
intelligence. These tasks include learning, reasoning, problem solving,
perception, and understanding language.


In [13]:
# Cell 8: Generate answers from retrieved context

def generate_answer(contexts):
    """
    Create an answer from the retrieved RAG context.
    """
    return " ".join(contexts[:2])

qa_df["response"] = qa_df["retrieved_contexts"].apply(generate_answer)

print("Answers generated successfully.")

print("\nExample:")
print("Question:", qa_df.loc[0, "question"])
print("Answer:", qa_df.loc[0, "response"])

Answers generated successfully.

Example:
Question: What is Artificial Intelligence?
Answer: Artificial Intelligence (AI) is a field of computer science that focuses on
creating systems that can perform tasks that normally require human
intelligence. These tasks include learning, reasoning, problem solving,
perception, and understanding language. Machine Learning (ML) is a subset of Artificial Intelligence. It enables
computers to learn patterns from data and make predictions or decisions
without being explicitly programmed for every individual task.


In [14]:
# Cell 9: Prepare and inspect evaluation data

evaluation_df = qa_df[
    ["question", "response", "reference", "retrieved_contexts"]
].copy()

evaluation_df.columns = [
    "user_input",
    "response",
    "reference",
    "retrieved_contexts"
]

print("Evaluation dataset shape:", evaluation_df.shape)

display(evaluation_df)

Evaluation dataset shape: (10, 4)


,user_input,response,reference,retrieved_contexts
0,What is Artificial Intelligence?,Artificial Intelligence (AI) is a field of com...,Artificial Intelligence is a field of computer...,[Artificial Intelligence (AI) is a field of co...
1,What is Machine Learning?,Machine Learning (ML) is a subset of Artificia...,Machine Learning is a subset of Artificial Int...,[Machine Learning (ML) is a subset of Artifici...
2,What is supervised learning?,Supervised learning is a machine learning appr...,Supervised learning is a machine learning appr...,[Supervised learning is a machine learning app...
3,What is unsupervised learning?,Unsupervised learning uses data without labell...,Unsupervised learning uses data without labell...,[Unsupervised learning uses data without label...
4,What is Deep Learning?,Deep Learning is a branch of machine learning ...,Deep Learning is a branch of machine learning ...,[Deep Learning is a branch of machine learning...
5,What is Natural Language Processing?,Natural Language Processing (NLP) is a field o...,Natural Language Processing is a field of Arti...,[Natural Language Processing (NLP) is a field ...
6,What are Large Language Models?,Large Language Models (LLMs) are machine learn...,Large Language Models are machine learning mod...,[Large Language Models (LLMs) are machine lear...
7,What is Retrieval-Augmented Generation?,Retrieval-Augmented Generation (RAG) combines ...,Retrieval-Augmented Generation combines inform...,[Retrieval-Augmented Generation (RAG) combines...
8,What is a vector database?,Vector databases store numerical representatio...,A vector database stores numerical representat...,[Vector databases store numerical representati...
9,What is ChromaDB?,ChromaDB is a vector database commonly used in...,ChromaDB is a vector database commonly used in...,[ChromaDB is a vector database commonly used i...


In [15]:
# Cell 10: Validate evaluation data before Ragas

print("Checking evaluation dataset...\n")

print("Rows:", len(evaluation_df))
print("Missing user_input:", evaluation_df["user_input"].isna().sum())
print("Missing response:", evaluation_df["response"].isna().sum())
print("Missing reference:", evaluation_df["reference"].isna().sum())
print("Missing retrieved_contexts:", evaluation_df["retrieved_contexts"].isna().sum())

empty_contexts = sum(
    1 for x in evaluation_df["retrieved_contexts"]
    if not x or len(x) == 0
)

print("Empty context lists:", empty_contexts)

if (
    evaluation_df["user_input"].isna().sum() == 0
    and evaluation_df["response"].isna().sum() == 0
    and evaluation_df["reference"].isna().sum() == 0
    and empty_contexts == 0
):
    print("\n✓ Dataset is ready for Ragas evaluation.")
else:
    print("\n✗ Dataset has a problem. Do not run Ragas yet.")

Checking evaluation dataset...

Rows: 10
Missing user_input: 0
Missing response: 0
Missing reference: 0
Missing retrieved_contexts: 0
Empty context lists: 0

✓ Dataset is ready for Ragas evaluation.


In [16]:
# Cell 11: Install local Hugging Face evaluator dependencies

!pip install -q -U transformers accelerate torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.6/554.6 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.1/553.1 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2

In [17]:
# Cell 12: Import Hugging Face evaluator libraries

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.11.0+cpu
CUDA available: False


In [20]:
# Cell 13: Load the local evaluation model

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
)

print("Local evaluation model loaded successfully.")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Local evaluation model loaded successfully.


In [21]:
# Cell 14: Test the local evaluator

test_prompt = """
Answer the following question using the given context.

Context:
Artificial Intelligence is a field of computer science that focuses on
creating systems capable of tasks that normally require human intelligence.

Question:
What is Artificial Intelligence?

Answer:
"""

inputs = tokenizer(
    test_prompt,
    return_tensors="pt",
    truncation=True
)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100
    )

test_answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Evaluator test output:")
print(test_answer)

Evaluator test output:
field of computer science


In [22]:
# Cell 15: Final pre-Ragas validation

print("Final Ragas input check")
print("=" * 40)

print("Number of samples:", len(evaluation_df))
print("Columns:", list(evaluation_df.columns))

for column in evaluation_df.columns:
    print(
        f"{column}:",
        evaluation_df[column].notna().sum(),
        "non-empty rows"
    )

print("\nSample:")
print("Question:", evaluation_df.iloc[0]["user_input"])
print("Reference:", evaluation_df.iloc[0]["reference"])
print("Response:", evaluation_df.iloc[0]["response"])
print("Contexts:", len(evaluation_df.iloc[0]["retrieved_contexts"]))

print("\n✓ Ready for the Ragas evaluation stage.")

Final Ragas input check
Number of samples: 10
Columns: ['user_input', 'response', 'reference', 'retrieved_contexts']
user_input: 10 non-empty rows
response: 10 non-empty rows
reference: 10 non-empty rows
retrieved_contexts: 10 non-empty rows

Sample:
Question: What is Artificial Intelligence?
Reference: Artificial Intelligence is a field of computer science that creates systems capable of tasks that normally require human intelligence.
Response: Artificial Intelligence (AI) is a field of computer science that focuses on
creating systems that can perform tasks that normally require human
intelligence. These tasks include learning, reasoning, problem solving,
perception, and understanding language. Machine Learning (ML) is a subset of Artificial Intelligence. It enables
computers to learn patterns from data and make predictions or decisions
without being explicitly programmed for every individual task.
Contexts: 3

✓ Ready for the Ragas evaluation stage.


In [24]:
# Cell 16: Fix Ragas compatibility

!pip install -q "langchain-community==0.4.1"

print("Compatible langchain-community version installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.0 MB/s eta 0:00:00
Compatible langchain-community version installed.


In [25]:
# Cell 17: Import Ragas after compatibility fix

import ragas

print("Ragas version:", ragas.__version__)

from ragas import metrics

print("\nRagas imported successfully.")
print("\nRequired metrics:")
print("Faithfulness:", hasattr(metrics, "Faithfulness"))
print("Answer Relevancy:", hasattr(metrics, "AnswerRelevancy"))
print("Context Precision:", hasattr(metrics, "ContextPrecision"))
print("Context Recall:", hasattr(metrics, "ContextRecall"))

Ragas version: 0.4.3

Ragas imported successfully.

Required metrics:
Faithfulness: True
Answer Relevancy: True
Context Precision: True
Context Recall: True


/tmp/ipykernel_1385/3447524031.py:11: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  print("Faithfulness:", hasattr(metrics, "Faithfulness"))
/tmp/ipykernel_1385/3447524031.py:12: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  print("Answer Relevancy:", hasattr(metrics, "AnswerRelevancy"))
/tmp/ipykernel_1385/3447524031.py:13: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  print("Context Precision:", hasattr(metrics, "ContextPrecision"))
/tmp/ipykernel_1385/3447524031.py:

In [26]:
# Cell 18: Create Ragas LLM wrapper

from ragas.llms import LangchainLLMWrapper
from langchain_core.language_models.llms import LLM

class LocalFlanT5LLM(LLM):
    @property
    def _llm_type(self):
        return "local-flan-t5"

    def _call(self, prompt, stop=None, run_manager=None, **kwargs):
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=512
        )

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=150
            )

        return tokenizer.decode(
            outputs[0],
            skip_special_tokens=True
        )

local_llm = LocalFlanT5LLM()

ragas_llm = LangchainLLMWrapper(local_llm)

print("Ragas LLM wrapper created successfully.")

Ragas LLM wrapper created successfully.


/tmp/ipykernel_1385/2204336441.py:32: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(local_llm)


In [27]:
# Cell 19: Test the Ragas LLM wrapper

test_prompt = """
Answer in one short sentence.

Question:
What is Machine Learning?

Answer:
"""

test_result = local_llm.invoke(test_prompt)

print("Ragas LLM test output:")
print(test_result)

Ragas LLM test output:
Machine learning is a technology that uses machine learning to learn information.


In [28]:
# Cell 20: Generate answers using retrieved context

def generate_rag_answer(row):
    context = "\n".join(row["retrieved_contexts"])

    prompt = f"""
Answer the question using only the information in the context.

Context:
{context}

Question:
{row["user_input"]}

Give a short, clear answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100
        )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    ).strip()


evaluation_df["response"] = evaluation_df.apply(
    generate_rag_answer,
    axis=1
)

print("Generated answers:", len(evaluation_df))
print("\nExample")
print("Question:", evaluation_df.iloc[0]["user_input"])
print("Answer:", evaluation_df.iloc[0]["response"])

Generated answers: 10

Example
Question: What is Artificial Intelligence?
Answer: field of computer science


In [29]:
# Cell 21: Create Ragas EvaluationDataset

from ragas import EvaluationDataset

ragas_dataset = EvaluationDataset.from_pandas(evaluation_df)

print("Ragas dataset created successfully.")
print("Number of samples:", len(ragas_dataset))

Ragas dataset created successfully.
Number of samples: 10


In [30]:
# Cell 22: Verify Ragas dataset

print("First evaluation sample:")
print(ragas_dataset[0])

First evaluation sample:
user_input='What is Artificial Intelligence?' retrieved_contexts=['Artificial Intelligence (AI) is a field of computer science that focuses on\ncreating systems that can perform tasks that normally require human\nintelligence. These tasks include learning, reasoning, problem solving,\nperception, and understanding language.', 'Machine Learning (ML) is a subset of Artificial Intelligence. It enables\ncomputers to learn patterns from data and make predictions or decisions\nwithout being explicitly programmed for every individual task.', 'Natural Language Processing (NLP) is a field of Artificial Intelligence that\nallows computers to process and understand human language. Applications\ninclude translation, sentiment analysis, chatbots, and text summarization.'] reference_contexts=None retrieved_context_ids=None reference_context_ids=None response='field of computer science' multi_responses=None reference='Artificial Intelligence is a field of computer science tha

In [33]:
# Cell 23: Configure Ragas metrics for Ragas 0.4.3

from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)

faithfulness.llm = ragas_llm
answer_relevancy.llm = ragas_llm
context_precision.llm = ragas_llm
context_recall.llm = ragas_llm

metrics_list = [
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
]

print("All four Ragas metrics configured successfully.")

All four Ragas metrics configured successfully.


/tmp/ipykernel_1385/948602153.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/tmp/ipykernel_1385/948602153.py:3: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
/tmp/ipykernel_1385/948602153.py:3: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
/tmp/ipykernel_1385/948602153.py:3: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in 

In [34]:
# Cell 24: Verify metric configuration

for metric in metrics_list:
    print(
        type(metric).__name__,
        "→ LLM configured:",
        metric.llm is not None
    )

print("\n✓ Ready for evaluation.")

Faithfulness → LLM configured: True
AnswerRelevancy → LLM configured: True
ContextPrecision → LLM configured: True
ContextRecall → LLM configured: True

✓ Ready for evaluation.


In [36]:
# Cell 25: Configure Ragas with our existing Hugging Face embeddings

from ragas.embeddings import LangchainEmbeddingsWrapper

ragas_embeddings = LangchainEmbeddingsWrapper(
    embedding_model
)

print("Ragas embeddings configured successfully.")

Ragas embeddings configured successfully.


/tmp/ipykernel_1385/3898603809.py:5: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(


In [37]:
# Cell 26: Test Ragas embeddings

test_embedding = ragas_embeddings.embed_query(
    "What is Machine Learning?"
)

print("Embedding generated successfully.")
print("Embedding length:", len(test_embedding))

Embedding generated successfully.
Embedding length: 384


In [40]:
# Cell 27: Check Ragas evaluator support

import inspect
from ragas.llms import llm_factory

print("llm_factory available:", callable(llm_factory))

print("\nllm_factory signature:")
print(inspect.signature(llm_factory))

print("\nCurrent Ragas version:", ragas.__version__)

llm_factory available: True

llm_factory signature:
(model: 'str', provider: 'str' = 'openai', client: 't.Optional[t.Any]' = None, adapter: 'str' = 'auto', cache: 't.Optional[CacheInterface]' = None, **kwargs: 't.Any') -> 'InstructorBaseRagasLLM'

Current Ragas version: 0.4.3


In [41]:
# Cell 28: Check for an OpenAI API key

import os

openai_key = os.environ.get("OPENAI_API_KEY")

if openai_key:
    print("OPENAI_API_KEY is available.")
else:
    print("OPENAI_API_KEY is NOT available.")

OPENAI_API_KEY is NOT available.


In [42]:
# Cell 29: Check Ragas LLM adapter options

from ragas.llms import llm_factory
import inspect

print("llm_factory parameters:")
print(inspect.signature(llm_factory))

print("\nSupported provider-related information:")
print(llm_factory.__doc__)

llm_factory parameters:
(model: 'str', provider: 'str' = 'openai', client: 't.Optional[t.Any]' = None, adapter: 'str' = 'auto', cache: 't.Optional[CacheInterface]' = None, **kwargs: 't.Any') -> 'InstructorBaseRagasLLM'

Supported provider-related information:

Create an LLM instance for structured output generation with automatic adapter selection.

Supports multiple LLM providers and structured output backends with unified interface
for both sync and async operations. Returns instances with .generate() and .agenerate()
methods that accept Pydantic models for structured outputs.

Auto-detects the best adapter for your provider:
- Google Gemini → uses LiteLLM adapter
- Other providers → uses Instructor adapter (default)
- Explicit control available via adapter parameter

Args:
    model: Model name (e.g., "gpt-4o", "claude-3-sonnet", "gemini-2.0-flash").
    provider: LLM provider (default: "openai").
             Examples: openai, anthropic, google, groq, mistral, etc.
    client: Pre-

In [43]:
# Cell 30: Check for a Google Gemini API key

import os

gemini_key = os.environ.get("GOOGLE_API_KEY")

if gemini_key:
    print("GOOGLE_API_KEY is available.")
else:
    print("GOOGLE_API_KEY is NOT available.")

GOOGLE_API_KEY is NOT available.


In [44]:
# Cell 31: Check available Ragas evaluator options

from ragas.llms import llm_factory
import inspect

print("Ragas version:", ragas.__version__)
print("\nSupported llm_factory signature:")
print(inspect.signature(llm_factory))

print("\n✓ Ragas structured evaluator API is available.")

Ragas version: 0.4.3

Supported llm_factory signature:
(model: 'str', provider: 'str' = 'openai', client: 't.Optional[t.Any]' = None, adapter: 'str' = 'auto', cache: 't.Optional[CacheInterface]' = None, **kwargs: 't.Any') -> 'InstructorBaseRagasLLM'

✓ Ragas structured evaluator API is available.


In [45]:
# Cell 32: Test structured evaluator output

structured_prompt = """
You are an evaluation system.

Evaluate whether the answer is supported by the context.

Context:
Artificial Intelligence is a field of computer science that creates systems
capable of tasks that normally require human intelligence.

Question:
What is Artificial Intelligence?

Answer:
Artificial Intelligence is a field of computer science.

Return ONLY this JSON:
{"verdict": 1, "reason": "The answer is supported by the context."}
"""

inputs = tokenizer(
    structured_prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80
    )

structured_result = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Model structured-output test:")
print(structured_result)

Model structured-output test:
yes


In [46]:
from google.colab import userdata

groq_api_key = userdata.get("GROQ_API_KEY")

print("Groq API key loaded:", bool(groq_api_key))

Groq API key loaded: True


In [47]:
# Cell 33: Install Groq SDK

!pip install -q -U groq

print("Groq SDK installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 2.6 MB/s eta 0:00:00
Groq SDK installed.


In [48]:
# Cell 34: Connect to Groq

import os
from google.colab import userdata
from groq import Groq

groq_api_key = userdata.get("GROQ_API_KEY")

if not groq_api_key:
    raise ValueError("GROQ_API_KEY was not found in Colab Secrets.")

os.environ["GROQ_API_KEY"] = groq_api_key

groq_client = Groq(api_key=groq_api_key)

print("Groq API key loaded successfully.")
print("Groq client created successfully.")

Groq API key loaded successfully.
Groq client created successfully.


In [52]:
# Cell 35: Install LiteLLM

!pip install -q -U litellm

print("LiteLLM installed successfully.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.1/28.1 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.3/278.3 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.9/15.9 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 7.2 MB/s eta 0:00:00


Streaming output truncated to the last 5000 lines.

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/ragas/prompt/pydantic_prompt.py", line 537, in parse_output_string
    result = super().parse(jsonstr)
  File "/usr/local/lib/python3.13/dist-packages/langchain_core/output_parsers/pydantic.py", line 93, in parse
    return self.parse_result([Generation(text=text)])
           ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/langchain_core/output_parsers/pydantic.py", line 78, in parse_result
    return self._parse_obj(json_object)
           ~~~~~~~~~~~~~~~^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/langchain_core/output_parsers/pydantic.py", line 37, in _parse_obj
    raise self._parser_exception(e, obj) from e
langchain_core.exceptions.OutputParserException: Failed to parse StringIO from completion "output_string". Got: 1 

LiteLLM installed successfully.


In [53]:
# Cell 36: Create Groq evaluator through LiteLLM

import litellm
from ragas.llms import llm_factory

ragas_groq_llm = llm_factory(
    "groq/openai/gpt-oss-20b",
    provider="litellm",
    client=litellm.completion
)

print("Ragas Groq evaluator created successfully.")

Ragas Groq evaluator created successfully.


In [54]:
# Cell 37: Test Ragas structured output

from pydantic import BaseModel, Field

class TestEvaluation(BaseModel):
    verdict: int = Field(description="1 if the answer is supported, otherwise 0")
    reason: str = Field(description="Brief explanation")

test_prompt = """
Evaluate this answer.

Context:
Artificial Intelligence is a field of computer science that creates systems
capable of tasks that normally require human intelligence.

Question:
What is Artificial Intelligence?

Answer:
Artificial Intelligence is a field of computer science.

Return the verdict and reason.
"""

test_result = ragas_groq_llm.generate(
    test_prompt,
    TestEvaluation
)

print("Structured evaluator test:")
print(test_result)

Structured evaluator test:
verdict=0 reason='The answer omits the key aspect of AI—creating systems capable of tasks that normally require human intelligence—making it incomplete.'


In [55]:
# Cell 38: Configure Ragas metrics with Groq evaluator

from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)

faithfulness.llm = ragas_groq_llm
answer_relevancy.llm = ragas_groq_llm
context_precision.llm = ragas_groq_llm
context_recall.llm = ragas_groq_llm

metrics_list = [
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
]

print("Four Ragas metrics configured:")
print("✓ Faithfulness")
print("✓ Answer Relevancy")
print("✓ Context Precision")
print("✓ Context Recall")

Four Ragas metrics configured:
✓ Faithfulness
✓ Answer Relevancy
✓ Context Precision
✓ Context Recall


/tmp/ipykernel_1385/2115249227.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/tmp/ipykernel_1385/2115249227.py:3: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
/tmp/ipykernel_1385/2115249227.py:3: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
/tmp/ipykernel_1385/2115249227.py:3: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed

In [57]:
# Cell 39: Run Ragas evaluation with reduced token usage

from ragas import evaluate
from ragas.run_config import RunConfig

# Reduce Ragas generation count where supported
for metric in metrics_list:
    if hasattr(metric, "max_retries"):
        metric.max_retries = 1

run_config = RunConfig(
    timeout=120,
    max_retries=1,
    max_wait=10
)

result = evaluate(
    dataset=ragas_dataset,
    metrics=metrics_list,
    llm=ragas_groq_llm,
    embeddings=ragas_embeddings,
    run_config=run_config,
    batch_size=1,
    raise_exceptions=False,
    show_progress=True
)

print("Evaluation finished.")

Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

Batch 1/40:   0%|          | 0/1 [00:00<?, ?it/s]

ERROR:instructor.v2.retry:API call failed on attempt 1: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m3kaq9xjet691q8nydtp1ety` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7662, Requested 924. Please try again in 4.395s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}

ERROR:instructor.v2.retry:Max retries exceeded. Total attempts: 1, Last error: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m3kaq9xjet691q8nydtp1ety` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7662, Requested 924. Please try again in 4.395s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}



Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



ERROR:instructor.v2.retry:API call failed on attempt 1: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m3kaq9xjet691q8nydtp1ety` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7633, Requested 928. Please try again in 4.2075s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}

ERROR:instructor.v2.retry:Max retries exceeded. Total attempts: 1, Last error: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m3kaq9xjet691q8nydtp1ety` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7633, Requested 928. Please try again in 4.2075s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}

...


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



ERROR:instructor.v2.retry:API call failed on attempt 1: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m3kaq9xjet691q8nydtp1ety` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7084, Requested 1528. Please try again in 4.59s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}

ERROR:instructor.v2.retry:Max retries exceeded. Total attempts: 1, Last error: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m3kaq9xjet691q8nydtp1ety` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7084, Requested 1528. Please try again in 4.59s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}



Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



ERROR:instructor.v2.retry:API call failed on attempt 1: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m3kaq9xjet691q8nydtp1ety` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7042, Requested 1662. Please try again in 5.279999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}

ERROR:instructor.v2.retry:Max retries exceeded. Total attempts: 1, Last error: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m3kaq9xjet691q8nydtp1ety` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7042, Requested 1662. Please try again in 5.279999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limi


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

Evaluation finished.


In [58]:
# Cell 42: Check Ragas metric generation settings

for metric in metrics_list:
    print(
        type(metric).__name__,
        "| strictness:",
        getattr(metric, "strictness", "not available"),
        "| max_retries:",
        getattr(metric, "max_retries", "not available")
    )

Faithfulness | strictness: not available | max_retries: 1
AnswerRelevancy | strictness: 3 | max_retries: not available
ContextPrecision | strictness: not available | max_retries: 1
ContextRecall | strictness: not available | max_retries: 1


In [59]:
# Cell 43: Reduce Answer Relevancy generation count

answer_relevancy.strictness = 1

print("AnswerRelevancy strictness:", answer_relevancy.strictness)
print("Faithfulness max_retries:", faithfulness.max_retries)
print("ContextPrecision max_retries:", context_precision.max_retries)
print("ContextRecall max_retries:", context_recall.max_retries)

AnswerRelevancy strictness: 1
Faithfulness max_retries: 1
ContextPrecision max_retries: 1
ContextRecall max_retries: 1


In [61]:
# Cell 44: Test one sample with Faithfulness

from ragas import evaluate
from ragas.run_config import RunConfig

one_sample_dataset = EvaluationDataset.from_pandas(
    evaluation_df.iloc[:1]
)

test_run_config = RunConfig(
    timeout=120,
    max_retries=1,
    max_wait=10
)

test_result = evaluate(
    dataset=one_sample_dataset,
    metrics=[faithfulness],
    llm=ragas_groq_llm,
    embeddings=ragas_embeddings,
    run_config=test_run_config,
    batch_size=1,
    raise_exceptions=True,
    show_progress=True
)

print("\nSingle-sample Faithfulness test completed.")
print(test_result.to_pandas())

Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

Batch 1/1:   0%|          | 0/1 [00:00<?, ?it/s]


Single-sample Faithfulness test completed.
                         user_input  \
0  What is Artificial Intelligence?   

                                  retrieved_contexts  \
0  [Artificial Intelligence (AI) is a field of co...   

                    response  \
0  field of computer science   

                                           reference  faithfulness  
0  Artificial Intelligence is a field of computer...           1.0  


In [62]:
# Cell 45: Full Ragas evaluation

from ragas import evaluate
from ragas.run_config import RunConfig

full_run_config = RunConfig(
    timeout=120,
    max_retries=1,
    max_wait=10
)

result = evaluate(
    dataset=ragas_dataset,
    metrics=metrics_list,
    llm=ragas_groq_llm,
    embeddings=ragas_embeddings,
    run_config=full_run_config,
    batch_size=1,
    raise_exceptions=False,
    show_progress=True
)

print("\nFull Ragas evaluation completed.")

Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

Batch 1/40:   0%|          | 0/1 [00:00<?, ?it/s]

ERROR:instructor.v2.retry:API call failed on attempt 1: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m3kaq9xjet691q8nydtp1ety` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7137, Requested 938. Please try again in 562.5ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}

ERROR:instructor.v2.retry:Max retries exceeded. Total attempts: 1, Last error: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m3kaq9xjet691q8nydtp1ety` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7137, Requested 938. Please try again in 562.5ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



ERROR:instructor.v2.retry:API call failed on attempt 1: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m3kaq9xjet691q8nydtp1ety` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7103, Requested 1542. Please try again in 4.8375s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}

ERROR:instructor.v2.retry:Max retries exceeded. Total attempts: 1, Last error: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m3kaq9xjet691q8nydtp1ety` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7103, Requested 1542. Please try again in 4.8375s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded

...


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



ERROR:instructor.v2.retry:API call failed on attempt 1: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m3kaq9xjet691q8nydtp1ety` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7113, Requested 937. Please try again in 375ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}

ERROR:instructor.v2.retry:Max retries exceeded. Total attempts: 1, Last error: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m3kaq9xjet691q8nydtp1ety` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7113, Requested 937. Please try again in 375ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}

E


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



ERROR:instructor.v2.retry:API call failed on attempt 1: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m3kaq9xjet691q8nydtp1ety` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7076, Requested 1655. Please try again in 5.4825s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}

ERROR:instructor.v2.retry:Max retries exceeded. Total attempts: 1, Last error: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m3kaq9xjet691q8nydtp1ety` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7076, Requested 1655. Please try again in 5.4825s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Full Ragas evaluation completed.


In [63]:
# Cell 46: Reduce Ragas parallelism

from ragas.run_config import RunConfig

sequential_run_config = RunConfig(
    timeout=120,
    max_retries=1,
    max_wait=10,
    max_workers=1
)

print("Ragas evaluation will use:")
print("max_workers:", sequential_run_config.max_workers)
print("max_retries:", sequential_run_config.max_retries)

Ragas evaluation will use:
max_workers: 1
max_retries: 1


In [69]:
# Cell 47: Safe single-sample Ragas evaluation

import time
from ragas import evaluate
from ragas.run_config import RunConfig

safe_run_config = RunConfig(
    timeout=120,
    max_retries=1,
    max_wait=10,
    max_workers=1
)

def evaluate_one_sample(sample_df, metric):
    sample_dataset = EvaluationDataset.from_pandas(sample_df)

    result = evaluate(
        dataset=sample_dataset,
        metrics=[metric],
        llm=ragas_groq_llm,
        embeddings=ragas_embeddings,
        run_config=safe_run_config,
        batch_size=1,
        raise_exceptions=True,
        show_progress=False
    )

    return result.to_pandas()

print("Safe single-sample evaluator created.")

Safe single-sample evaluator created.


In [70]:
# Cell 48: Test safe single-sample evaluation

print("Testing Faithfulness on sample 1...")

sample_1_result = evaluate_one_sample(
    evaluation_df.iloc[:1],
    faithfulness
)

print("\nResult:")
display(sample_1_result)

print("\nWaiting for Groq rate limit window...")
time.sleep(10)

print("Test completed.")

Testing Faithfulness on sample 1...

Result:


,user_input,retrieved_contexts,response,reference,faithfulness
0,What is Artificial Intelligence?,[Artificial Intelligence (AI) is a field of co...,field of computer science,Artificial Intelligence is a field of computer...,1.0



Waiting for Groq rate limit window...
Test completed.


In [71]:
# Cell 49: Evaluate Faithfulness for all 10 samples safely

faithfulness_scores = []

for i in range(len(evaluation_df)):
    print(f"Evaluating Faithfulness: sample {i + 1}/10")

    try:
        sample_result = evaluate_one_sample(
            evaluation_df.iloc[i:i+1],
            faithfulness
        )

        score = sample_result["faithfulness"].iloc[0]
        faithfulness_scores.append(score)

        print("Score:", score)

    except Exception as e:
        print("Error:", type(e).__name__)
        faithfulness_scores.append(float("nan"))

    # Wait between API calls to stay below Groq TPM limit
    if i < len(evaluation_df) - 1:
        time.sleep(10)

evaluation_df["faithfulness"] = faithfulness_scores

print("\nFaithfulness baseline completed.")
print(evaluation_df[["user_input", "faithfulness"]])
print("\nMean Faithfulness:",
      evaluation_df["faithfulness"].mean())

Evaluating Faithfulness: sample 1/10
Score: 1.0
Evaluating Faithfulness: sample 2/10
Score: 1.0
Evaluating Faithfulness: sample 3/10
Score: 1.0
Evaluating Faithfulness: sample 4/10
Score: 1.0
Evaluating Faithfulness: sample 5/10
Score: 1.0
Evaluating Faithfulness: sample 6/10
Score: 1.0
Evaluating Faithfulness: sample 7/10
Score: 1.0
Evaluating Faithfulness: sample 8/10
Score: 1.0
Evaluating Faithfulness: sample 9/10
Score: 1.0
Evaluating Faithfulness: sample 10/10
Score: 1.0

Faithfulness baseline completed.
                                user_input  faithfulness
0         What is Artificial Intelligence?           1.0
1                What is Machine Learning?           1.0
2             What is supervised learning?           1.0
3           What is unsupervised learning?           1.0
4                   What is Deep Learning?           1.0
5     What is Natural Language Processing?           1.0
6          What are Large Language Models?           1.0
7  What is Retrieval-Augmente

In [72]:
# Cell 50: Check Faithfulness baseline

print("Faithfulness scores:")
print(evaluation_df["faithfulness"].tolist())

print("\nMean Faithfulness:",
      evaluation_df["faithfulness"].mean())

print("\nMissing scores:",
      evaluation_df["faithfulness"].isna().sum())

Faithfulness scores:
[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]

Mean Faithfulness: 1.0

Missing scores: 0


In [73]:
# Cell 51: Evaluate Answer Relevancy for all 10 samples safely

answer_relevancy_scores = []

for i in range(len(evaluation_df)):
    print(f"Evaluating Answer Relevancy: sample {i + 1}/10")

    try:
        sample_result = evaluate_one_sample(
            evaluation_df.iloc[i:i+1],
            answer_relevancy
        )

        score = sample_result["answer_relevancy"].iloc[0]
        answer_relevancy_scores.append(score)

        print("Score:", score)

    except Exception as e:
        print("Error:", type(e).__name__)
        answer_relevancy_scores.append(float("nan"))

    if i < len(evaluation_df) - 1:
        time.sleep(10)

evaluation_df["answer_relevancy"] = answer_relevancy_scores

print("\nAnswer Relevancy baseline completed.")
print(evaluation_df[["user_input", "answer_relevancy"]])

print("\nMean Answer Relevancy:",
      evaluation_df["answer_relevancy"].mean())

Evaluating Answer Relevancy: sample 1/10
Score: 0.5120077068160634
Evaluating Answer Relevancy: sample 2/10
Score: 0.0
Evaluating Answer Relevancy: sample 3/10
Score: 0.4999648467492943
Evaluating Answer Relevancy: sample 4/10
Score: 0.35313842440595605
Evaluating Answer Relevancy: sample 5/10
Score: 0.0
Evaluating Answer Relevancy: sample 6/10
Score: 0.0
Evaluating Answer Relevancy: sample 7/10
Score: 0.63560224830579
Evaluating Answer Relevancy: sample 8/10
Score: 0.2321777787908666
Evaluating Answer Relevancy: sample 9/10
Score: 0.8670510553713883
Evaluating Answer Relevancy: sample 10/10
Score: 0.231709738774557

Answer Relevancy baseline completed.
                                user_input  answer_relevancy
0         What is Artificial Intelligence?          0.512008
1                What is Machine Learning?          0.000000
2             What is supervised learning?          0.499965
3           What is unsupervised learning?          0.353138
4                   What is Deep 

In [74]:
# Cell 52: Check Answer Relevancy baseline

print("Answer Relevancy scores:")
print(evaluation_df["answer_relevancy"].tolist())

print("\nMean Answer Relevancy:",
      evaluation_df["answer_relevancy"].mean())

print("\nMissing scores:",
      evaluation_df["answer_relevancy"].isna().sum())

Answer Relevancy scores:
[0.5120077068160634, 0.0, 0.4999648467492943, 0.35313842440595605, 0.0, 0.0, 0.63560224830579, 0.2321777787908666, 0.8670510553713883, 0.231709738774557]

Mean Answer Relevancy: 0.3331651799213916

Missing scores: 0


In [78]:
# Cell 53: Evaluate Context Precision for all 10 samples

context_precision_scores = []

for i in range(len(evaluation_df)):
    print(f"Evaluating Context Precision: sample {i + 1}/10")

    try:
        sample_result = evaluate_one_sample(
            evaluation_df.iloc[i:i+1],
            context_precision
        )

        score = sample_result["context_precision"].iloc[0]
        context_precision_scores.append(score)

        print("Score:", score)

    except Exception as e:
        print("Error:", type(e).__name__)
        context_precision_scores.append(float("nan"))

    if i < len(evaluation_df) - 1:
        time.sleep(10)

evaluation_df["context_precision"] = context_precision_scores

print("\nContext Precision baseline completed.")
print(evaluation_df[["user_input", "context_precision"]])

print("\nMean Context Precision:",
      evaluation_df["context_precision"].mean())

Evaluating Context Precision: sample 1/10
Score: 0.9999999999
Evaluating Context Precision: sample 2/10
Score: 0.9999999999
Evaluating Context Precision: sample 3/10


ERROR:instructor.v2.retry:API call failed on attempt 1: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m3kaq9xjet691q8nydtp1ety` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7541, Requested 1674. Please try again in 9.1125s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}

ERROR:instructor.v2.retry:Max retries exceeded. Total attempts: 1, Last error: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m3kaq9xjet691q8nydtp1ety` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7541, Requested 1674. Please try again in 9.1125s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

Error: InstructorRetryException
Evaluating Context Precision: sample 4/10


ERROR:instructor.v2.retry:API call failed on attempt 1: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m3kaq9xjet691q8nydtp1ety` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 6741, Requested 1575. Please try again in 2.37s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}

ERROR:instructor.v2.retry:Max retries exceeded. Total attempts: 1, Last error: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m3kaq9xjet691q8nydtp1ety` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 6741, Requested 1575. Please try again in 2.37s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}



Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

Error: InstructorRetryException
Evaluating Context Precision: sample 5/10


...


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

Error: InstructorRetryException
Evaluating Context Precision: sample 7/10
Score: 0.9999999999
Evaluating Context Precision: sample 8/10


ERROR:instructor.v2.retry:API call failed on attempt 1: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m3kaq9xjet691q8nydtp1ety` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 6698, Requested 1595. Please try again in 2.1975s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}

ERROR:instructor.v2.retry:Max retries exceeded. Total attempts: 1, Last error: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m3kaq9xjet691q8nydtp1ety` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 6698, Requested 1595. Please try again in 2.1975s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

Error: InstructorRetryException
Evaluating Context Precision: sample 9/10
Score: 0.99999999995
Evaluating Context Precision: sample 10/10


ERROR:instructor.v2.retry:API call failed on attempt 1: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m3kaq9xjet691q8nydtp1ety` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 6676, Requested 1634. Please try again in 2.325s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}

ERROR:instructor.v2.retry:Max retries exceeded. Total attempts: 1, Last error: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m3kaq9xjet691q8nydtp1ety` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 6676, Requested 1634. Please try again in 2.325s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

Error: InstructorRetryException

Context Precision baseline completed.
                                user_input  context_precision
0         What is Artificial Intelligence?                1.0
1                What is Machine Learning?                1.0
2             What is supervised learning?                NaN
3           What is unsupervised learning?                NaN
4                   What is Deep Learning?                NaN
5     What is Natural Language Processing?                NaN
6          What are Large Language Models?                1.0
7  What is Retrieval-Augmented Generation?                NaN
8               What is a vector database?                1.0
9                        What is ChromaDB?                NaN

Mean Context Precision: 0.9999999999125


In [79]:
# Cell 54: Recalculate missing Context Precision scores safely

for i in range(len(evaluation_df)):

    if pd.isna(evaluation_df.loc[i, "context_precision"]):
        print(f"Re-evaluating Context Precision: sample {i + 1}/10")

        # Wait before making the API call
        time.sleep(20)

        try:
            sample_result = evaluate_one_sample(
                evaluation_df.iloc[i:i+1],
                context_precision
            )

            score = sample_result["context_precision"].iloc[0]
            evaluation_df.loc[i, "context_precision"] = score

            print("Score:", score)

        except Exception as e:
            print("Still failed:", type(e).__name__)

print("\nUpdated Context Precision:")
print(evaluation_df[["user_input", "context_precision"]])

print(
    "\nValid scores:",
    evaluation_df["context_precision"].notna().sum(),
    "/ 10"
)

print(
    "Mean:",
    evaluation_df["context_precision"].mean()
)

Re-evaluating Context Precision: sample 3/10
Score: 0.9999999999
Re-evaluating Context Precision: sample 4/10
Score: 0.9999999999
Re-evaluating Context Precision: sample 5/10
Score: 0.99999999995
Re-evaluating Context Precision: sample 6/10
Score: 0.9999999999
Re-evaluating Context Precision: sample 8/10
Score: 0.9999999999666667
Re-evaluating Context Precision: sample 10/10
Score: 0.99999999995

Updated Context Precision:
                                user_input  context_precision
0         What is Artificial Intelligence?                1.0
1                What is Machine Learning?                1.0
2             What is supervised learning?                1.0
3           What is unsupervised learning?                1.0
4                   What is Deep Learning?                1.0
5     What is Natural Language Processing?                1.0
6          What are Large Language Models?                1.0
7  What is Retrieval-Augmented Generation?                1.0
8              

In [80]:
# Cell 55: Verify Context Precision baseline

print("Context Precision scores:")
print(evaluation_df["context_precision"].tolist())

print("\nValid scores:",
      evaluation_df["context_precision"].notna().sum(), "/ 10")

print("Missing scores:",
      evaluation_df["context_precision"].isna().sum())

print("\nMean Context Precision:",
      evaluation_df["context_precision"].mean())

Context Precision scores:
[0.9999999999, 0.9999999999, 0.9999999999, 0.9999999999, 0.99999999995, 0.9999999999, 0.9999999999, 0.9999999999666667, 0.99999999995, 0.99999999995]

Valid scores: 10 / 10
Missing scores: 0

Mean Context Precision: 0.9999999999216665


In [81]:
# Cell 56: Evaluate Context Recall for all 10 samples safely

context_recall_scores = []

for i in range(len(evaluation_df)):
    print(f"Evaluating Context Recall: sample {i + 1}/10")

    # Wait before each API request
    time.sleep(20)

    try:
        sample_result = evaluate_one_sample(
            evaluation_df.iloc[i:i+1],
            context_recall
        )

        score = sample_result["context_recall"].iloc[0]
        context_recall_scores.append(score)

        print("Score:", score)

    except Exception as e:
        print("Error:", type(e).__name__)
        context_recall_scores.append(float("nan"))

evaluation_df["context_recall"] = context_recall_scores

print("\nContext Recall evaluation completed.")
print(evaluation_df[["user_input", "context_recall"]])

print("\nMean Context Recall:",
      evaluation_df["context_recall"].mean())

print("\nValid scores:",
      evaluation_df["context_recall"].notna().sum(), "/ 10")

Evaluating Context Recall: sample 1/10
Score: 1.0
Evaluating Context Recall: sample 2/10
Score: 1.0
Evaluating Context Recall: sample 3/10
Score: 1.0
Evaluating Context Recall: sample 4/10
Score: 1.0
Evaluating Context Recall: sample 5/10
Score: 1.0
Evaluating Context Recall: sample 6/10
Score: 1.0
Evaluating Context Recall: sample 7/10
Score: 1.0
Evaluating Context Recall: sample 8/10
Score: 1.0
Evaluating Context Recall: sample 9/10
Score: 1.0
Evaluating Context Recall: sample 10/10
Score: 1.0

Context Recall evaluation completed.
                                user_input  context_recall
0         What is Artificial Intelligence?             1.0
1                What is Machine Learning?             1.0
2             What is supervised learning?             1.0
3           What is unsupervised learning?             1.0
4                   What is Deep Learning?             1.0
5     What is Natural Language Processing?             1.0
6          What are Large Language Models?      

In [82]:
# Cell 57: Optimize retrieval by reducing k from 3 to 2

optimized_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

print("Original retrieval k: 3")
print("Optimized retrieval k: 2")
print("Optimized retriever created successfully.")

Original retrieval k: 3
Optimized retrieval k: 2
Optimized retriever created successfully.


In [83]:
# Cell 58: Retrieve optimized top-2 contexts

optimized_contexts = []

for question in evaluation_df["user_input"]:
    docs = optimized_retriever.invoke(question)
    contexts = [doc.page_content.strip() for doc in docs]
    optimized_contexts.append(contexts)

evaluation_df["optimized_contexts"] = optimized_contexts

print("Optimized contexts generated for all 10 questions.")

print("\nExample:")
print("Question:", evaluation_df.iloc[0]["user_input"])
print("\nOptimized contexts:")

for i, context in enumerate(evaluation_df.iloc[0]["optimized_contexts"], 1):
    print(f"\nContext {i}:")
    print(context)

Optimized contexts generated for all 10 questions.

Example:
Question: What is Artificial Intelligence?

Optimized contexts:

Context 1:
Artificial Intelligence (AI) is a field of computer science that focuses on
creating systems that can perform tasks that normally require human
intelligence. These tasks include learning, reasoning, problem solving,
perception, and understanding language.

Context 2:
Machine Learning (ML) is a subset of Artificial Intelligence. It enables
computers to learn patterns from data and make predictions or decisions
without being explicitly programmed for every individual task.


In [84]:
# Cell 59: Generate answers using optimized top-2 contexts

def generate_optimized_answer(row):
    context = "\n".join(row["optimized_contexts"])

    prompt = f"""
Answer the question using only the information in the context.

Context:
{context}

Question:
{row["user_input"]}

Give a short, clear and complete answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100
        )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    ).strip()

evaluation_df["optimized_response"] = evaluation_df.apply(
    generate_optimized_answer,
    axis=1
)

print("Optimized answers generated:", len(evaluation_df))

print("\nExample:")
print("Question:", evaluation_df.iloc[0]["user_input"])
print("Original answer:", evaluation_df.iloc[0]["response"])
print("Optimized answer:", evaluation_df.iloc[0]["optimized_response"])

Optimized answers generated: 10

Example:
Question: What is Artificial Intelligence?
Original answer: field of computer science
Optimized answer: field of computer science


In [85]:
# Cell 60: Prepare optimized Ragas evaluation dataset

optimized_evaluation_df = evaluation_df[
    ["user_input", "optimized_response", "reference", "optimized_contexts"]
].copy()

optimized_evaluation_df.columns = [
    "user_input",
    "response",
    "reference",
    "retrieved_contexts"
]

optimized_ragas_dataset = EvaluationDataset.from_pandas(
    optimized_evaluation_df
)

print("Optimized Ragas dataset created.")
print("Samples:", len(optimized_ragas_dataset))

print("\nExample:")
print(optimized_evaluation_df.iloc[0])

Optimized Ragas dataset created.
Samples: 10

Example:
user_input                             What is Artificial Intelligence?
response                                      field of computer science
reference             Artificial Intelligence is a field of computer...
retrieved_contexts    [Artificial Intelligence (AI) is a field of co...
Name: 0, dtype: object


In [86]:
# Cell 61: Evaluate optimized Answer Relevancy

optimized_answer_relevancy_scores = []

for i in range(len(optimized_evaluation_df)):
    print(f"Evaluating optimized Answer Relevancy: sample {i + 1}/10")

    time.sleep(20)

    try:
        sample_result = evaluate_one_sample(
            optimized_evaluation_df.iloc[i:i+1],
            answer_relevancy
        )

        score = sample_result["answer_relevancy"].iloc[0]
        optimized_answer_relevancy_scores.append(score)

        print("Score:", score)

    except Exception as e:
        print("Error:", type(e).__name__)
        optimized_answer_relevancy_scores.append(float("nan"))

optimized_evaluation_df["answer_relevancy"] = optimized_answer_relevancy_scores

print("\nOptimized Answer Relevancy completed.")

print(
    optimized_evaluation_df[
        ["user_input", "answer_relevancy"]
    ]
)

print(
    "\nOptimized Mean Answer Relevancy:",
    optimized_evaluation_df["answer_relevancy"].mean()
)

print(
    "Valid scores:",
    optimized_evaluation_df["answer_relevancy"].notna().sum(),
    "/ 10"
)

Evaluating optimized Answer Relevancy: sample 1/10
Score: 0.0
Evaluating optimized Answer Relevancy: sample 2/10
Score: 0.0
Evaluating optimized Answer Relevancy: sample 3/10
Score: 0.658032345848297
Evaluating optimized Answer Relevancy: sample 4/10
Score: 0.29569290710087714
Evaluating optimized Answer Relevancy: sample 5/10
Score: 0.0
Evaluating optimized Answer Relevancy: sample 6/10
Score: 0.0
Evaluating optimized Answer Relevancy: sample 7/10
Score: 0.63560224830579
Evaluating optimized Answer Relevancy: sample 8/10
Score: 0.2321777787908666
Evaluating optimized Answer Relevancy: sample 9/10
Score: 0.8670510553713883
Evaluating optimized Answer Relevancy: sample 10/10
Score: 0.231709738774557

Optimized Answer Relevancy completed.
                                user_input  answer_relevancy
0         What is Artificial Intelligence?          0.000000
1                What is Machine Learning?          0.000000
2             What is supervised learning?          0.658032
3        

In [87]:
# Cell 62: Optimize chunk size from 300 to 500

optimized_text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

optimized_chunks = optimized_text_splitter.split_documents(documents)

print("Original chunk size: 300")
print("Optimized chunk size: 500")
print("Original chunks:", len(chunks))
print("Optimized chunks:", len(optimized_chunks))

Original chunk size: 300
Optimized chunk size: 500
Original chunks: 10
Optimized chunks: 10


In [88]:
# Cell 63: Build optimized vector store with chunk size 500

optimized_vectorstore = Chroma.from_documents(
    documents=optimized_chunks,
    embedding=embedding_model,
    collection_name="w8d2_rag_optimized"
)

optimized_retriever_500 = optimized_vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

print("Optimized ChromaDB created successfully.")
print("Chunk size: 500")
print("Retrieval k: 3")

Optimized ChromaDB created successfully.
Chunk size: 500
Retrieval k: 3


In [89]:
# Cell 64: Retrieve contexts using optimized chunk size

optimized_contexts_500 = []

for question in evaluation_df["user_input"]:
    docs = optimized_retriever_500.invoke(question)
    contexts = [doc.page_content.strip() for doc in docs]
    optimized_contexts_500.append(contexts)

print("Optimized contexts generated for all 10 questions.")

print("\nExample:")
print("Question:", evaluation_df.iloc[0]["user_input"])

for i, context in enumerate(optimized_contexts_500[0], 1):
    print(f"\nContext {i}:")
    print(context)

Optimized contexts generated for all 10 questions.

Example:
Question: What is Artificial Intelligence?

Context 1:
Artificial Intelligence (AI) is a field of computer science that focuses on
creating systems that can perform tasks that normally require human
intelligence. These tasks include learning, reasoning, problem solving,
perception, and understanding language.

Context 2:
Machine Learning (ML) is a subset of Artificial Intelligence. It enables
computers to learn patterns from data and make predictions or decisions
without being explicitly programmed for every individual task.

Context 3:
Natural Language Processing (NLP) is a field of Artificial Intelligence that
allows computers to process and understand human language. Applications
include translation, sentiment analysis, chatbots, and text summarization.


In [90]:
# Cell 65: Generate answers using optimized contexts

def generate_optimized_answer_500(row):
    context = "\n".join(row["optimized_contexts_500"])

    prompt = f"""
Answer the question using only the information in the context.

Context:
{context}

Question:
{row["user_input"]}

Give a short, clear and complete answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100
        )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    ).strip()


evaluation_df["optimized_contexts_500"] = optimized_contexts_500

evaluation_df["optimized_response_500"] = evaluation_df.apply(
    generate_optimized_answer_500,
    axis=1
)

print("Optimized answers generated:", len(evaluation_df))

print("\nExample:")
print("Question:", evaluation_df.iloc[0]["user_input"])
print("Original answer:", evaluation_df.iloc[0]["response"])
print("Optimized answer:", evaluation_df.iloc[0]["optimized_response_500"])

Optimized answers generated: 10

Example:
Question: What is Artificial Intelligence?
Original answer: field of computer science
Optimized answer: field of computer science


In [91]:
# Cell 66: Prepare optimized dataset for Ragas evaluation

optimized_evaluation_df_500 = evaluation_df[
    ["user_input", "optimized_response_500", "reference", "optimized_contexts_500"]
].copy()

optimized_evaluation_df_500.columns = [
    "user_input",
    "response",
    "reference",
    "retrieved_contexts"
]

optimized_ragas_dataset_500 = EvaluationDataset.from_pandas(
    optimized_evaluation_df_500
)

print("Optimized Ragas dataset created successfully.")
print("Number of samples:", len(optimized_ragas_dataset_500))

print("\nExample:")
print(optimized_evaluation_df_500.iloc[0])

Optimized Ragas dataset created successfully.
Number of samples: 10

Example:
user_input                             What is Artificial Intelligence?
response                                      field of computer science
reference             Artificial Intelligence is a field of computer...
retrieved_contexts    [Artificial Intelligence (AI) is a field of co...
Name: 0, dtype: object


In [92]:
# Cell 67: Evaluate optimized Answer Relevancy

optimized_relevancy_results_500 = []

for i in range(len(optimized_evaluation_df_500)):
    print(f"Evaluating optimized Answer Relevancy: sample {i + 1}/10")

    sample_result = evaluate_one_sample(
        optimized_evaluation_df_500.iloc[[i]],
        answer_relevancy
    )

    score = sample_result["answer_relevancy"].iloc[0]
    optimized_relevancy_results_500.append(score)

    print("Score:", score)

    if i < len(optimized_evaluation_df_500) - 1:
        print("Waiting for Groq rate limit window...")
        time.sleep(20)

print("\nOptimized Answer Relevancy evaluation completed.")

Evaluating optimized Answer Relevancy: sample 1/10
Score: 0.0
Waiting for Groq rate limit window...
Evaluating optimized Answer Relevancy: sample 2/10
Score: 0.0
Waiting for Groq rate limit window...
Evaluating optimized Answer Relevancy: sample 3/10
Score: 0.4999648467492943
Waiting for Groq rate limit window...
Evaluating optimized Answer Relevancy: sample 4/10
Score: 0.0
Waiting for Groq rate limit window...
Evaluating optimized Answer Relevancy: sample 5/10
Score: 0.0
Waiting for Groq rate limit window...
Evaluating optimized Answer Relevancy: sample 6/10
Score: 0.0
Waiting for Groq rate limit window...
Evaluating optimized Answer Relevancy: sample 7/10
Score: 0.7163944058783666
Waiting for Groq rate limit window...
Evaluating optimized Answer Relevancy: sample 8/10
Score: 0.2321777787908666
Waiting for Groq rate limit window...
Evaluating optimized Answer Relevancy: sample 9/10
Score: 0.8670510553713883
Waiting for Groq rate limit window...
Evaluating optimized Answer Relevancy: s

In [93]:
# Cell 68: Test retrieval k = 1

optimized_retriever_k1 = vectorstore.as_retriever(
    search_kwargs={"k": 1}
)

print("Baseline retrieval k: 3")
print("Previous test retrieval k: 2")
print("New test retrieval k: 1")
print("k=1 retriever created successfully.")

Baseline retrieval k: 3
Previous test retrieval k: 2
New test retrieval k: 1
k=1 retriever created successfully.


In [94]:
# Cell 69: Generate contexts and answers with k = 1

k1_contexts = []

for question in evaluation_df["user_input"]:
    docs = optimized_retriever_k1.invoke(question)
    k1_contexts.append([doc.page_content.strip() for doc in docs])

evaluation_df["k1_contexts"] = k1_contexts


def generate_k1_answer(row):
    context = "\n".join(row["k1_contexts"])

    prompt = f"""
Answer the question using only the information in the context.

Context:
{context}

Question:
{row["user_input"]}

Give a short, clear and complete answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100
        )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    ).strip()


evaluation_df["k1_response"] = evaluation_df.apply(
    generate_k1_answer,
    axis=1
)

print("k=1 contexts and answers generated:", len(evaluation_df))

print("\nExample:")
print("Question:", evaluation_df.iloc[0]["user_input"])
print("k=1 answer:", evaluation_df.iloc[0]["k1_response"])

k=1 contexts and answers generated: 10

Example:
Question: What is Artificial Intelligence?
k=1 answer: field of computer science
